# Language Bias in Conflicting Multilingual RAG Evidence — Google Colab Runner

This notebook executes the full cross-lingual conflict experiment on a single Google Colab **NVIDIA T4 GPU (16 GB VRAM)** across:
- **`Qwen/Qwen2.5-7B-Instruct`**
- **`meta-llama/Llama-3.1-8B-Instruct`** (automatically falls back to the open mirror `unsloth/Meta-Llama-3.1-8B-Instruct` if `HF_TOKEN` is not set)

All commands run directly from the Colab workspace root using the cloned `nlp-hw/` directory prefix. All results, statistical tests, qualitative samples (`nlp-hw/results/experiment_results.json`), and LaTeX macros (`nlp-hw/docs/tex/tex/00_results_macros.tex`) are exported and packaged into `results_bundle.zip` for download at the end of the notebook.

In [ ]:
# 1. Verify T4 GPU & Clone Repository into nlp-hw/
!nvidia-smi
import os
if not os.path.exists('nlp-hw'):
    !git clone -b feat/hw-2 https://github.com/iacantek/nlp-hw.git
else:
    !git -C nlp-hw pull

In [ ]:
# 2. Install Dependencies via requirements.txt & Run Unit/Stress Tests
!pip install -q -r nlp-hw/requirements.txt
!pytest nlp-hw/src/tests/test_pipeline.py -v

In [ ]:
# 3. Optional: Set Hugging Face Token if using gated meta-llama/Llama-3.1-8B-Instruct
# (If left empty, the runner automatically uses the open mirror unsloth/Meta-Llama-3.1-8B-Instruct)
import os
try:
    from google.colab import userdata
    hf_tok = userdata.get('HF_TOKEN')
    if hf_tok:
        os.environ['HF_TOKEN'] = hf_tok
        print('Loaded HF_TOKEN from Colab Secrets.')
except Exception:
    pass

In [ ]:
# 4. Run Full Experiment for Qwen-2.5-7B-Instruct (4-bit NF4 on T4 GPU, batched --resume enabled)
!python3 nlp-hw/src/run_experiment.py --backend hf --models qwen --quantization 4bit --batch-size 16 --max-new-tokens 12 --seed 42 --output-dir nlp-hw/results --resume

In [ ]:
# Free GPU memory before loading the second 8B model
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# 5. Run Full Experiment for Llama-3.1-8B-Instruct & Export Combined Summary + LaTeX Macros
!python3 nlp-hw/src/run_experiment.py --backend hf --models qwen llama --quantization 4bit --batch-size 16 --max-new-tokens 12 --seed 42 --output-dir nlp-hw/results --resume --export-latex

In [ ]:
# 6. Package and Download Results Archive from Google Colab
!zip -r results_bundle.zip nlp-hw/results/ nlp-hw/docs/tex/tex/00_results_macros.tex
try:
    from google.colab import files
    files.download('results_bundle.zip')
except Exception as e:
    print('Archive ready at results_bundle.zip:', e)